
# Single-shot echo planar

The scope of this notebook is to take the segmentation of the previous page to
one shot, so that the whole matrix is acquired after a single excitation, and
to measure the two things that limit such an acquisition: the decay of the
signal over an echo train tens of milliseconds long, and the sensitivity of a
train of alternating readouts to a delay between the gradient and the
acquisition.

Outline:

#. **One excitation, every line.** The train, and the blip between its echoes.
#. **The trajectory.** What one shot traverses, and in what order.
#. **Decay across the train.** The point-spread function the decay produces
   along the phase-encode direction, against $T_2^*$.
#. **Gradient delay and the odd echoes.** The k-space displacement a delay
   introduces, measured with the analysis's own delay parameter.

The trade-off this page sits at one end of is measured in
:doc:`/generated/gallery/03-gre-to-epi/02_segmented`.


In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

PAGE_WIDTH = 8.6  # inches, the width of the documentation column

## One excitation, every line

The blip advances one line rather than the number of shots, and the train
runs the length of the matrix. Nothing else changes.


In [ ]:
import numpy as np

import pypulseqpp as pp

system = pp.Opts(
    max_grad=32.0,
    grad_unit="mT/m",
    max_slew=130.0,
    slew_unit="T/m/s",
    rf_dead_time=100e-6,
    rf_ringdown_time=20e-6,
    adc_dead_time=10e-6,
)

FOV = 220e-3
MATRIX = 64
THICKNESS = 5e-3
FLIP_ANGLE_DEG = 90.0
DWELL = 4e-6

rf, gz, gz_reph = pp.make_sinc_pulse(
    flip_angle=np.deg2rad(FLIP_ANGLE_DEG),
    duration=2e-3,
    slice_thickness=THICKNESS,
    apodization=0.5,
    time_bw_product=4.0,
    delay=system.rf_dead_time,
    system=system,
    use="excitation",
    return_gz=True,
)

acquisition = MATRIX * DWELL
raster = system.grad_raster_time
gx = pp.make_trapezoid(
    channel="x",
    amplitude=MATRIX / FOV / acquisition,
    flat_time=raster * np.ceil(acquisition / raster),
    system=system,
)
adc = pp.make_adc(num_samples=MATRIX, dwell=DWELL, delay=gx.rise_time, system=system)
gx_pre = pp.make_trapezoid(
    channel="x",
    area=-(gx.amplitude * gx.rise_time / 2 + (MATRIX / 2 + 0.5) / FOV),
    duration=5e-4,
    system=system,
)
gy_pre = pp.make_trapezoid(
    channel="y", area=-MATRIX / (2 * FOV), duration=5e-4, system=system
)
blip = pp.make_phase_blip(channel="y", fov=FOV, steps=1, system=system)

seq = pp.Sequence(system=system)
seq.add_block(rf, gz)
seq.add_block(gx_pre, gy_pre, gz_reph)
for echo in range(MATRIX):
    readout = pp.scale_grad(gx, (-1.0) ** echo)
    if echo == 0:
        seq.add_block(readout, adc)
    else:
        seq.add_block(readout, blip, adc)

ok, errors = seq.check_timing()
echo_spacing = pp.calc_duration(gx)
print(
    f"timing {ok}, {seq.num_blocks} blocks, "
    f"echo spacing {1e3 * echo_spacing:.3f} ms, "
    f"train {1e3 * MATRIX * echo_spacing:.1f} ms, "
    f"blip {1e6 * pp.calc_duration(blip):.0f} us"
)

seq.paper_plot()

## The trajectory

One shot, coloured by the rank of each echo in the train: the acquisition
starts at one corner of k-space and works across it, reversing direction at
every line.


In [ ]:
pp.plot.plot_kspace(seq, color_by="echo", plane="xy")

## Decay across the train

The last line is acquired tens of milliseconds after the first, and the
transverse signal has decayed over that interval. Along the phase-encode
direction the acquired data are therefore the true k-space multiplied by a
decaying envelope, and the image is convolved with that envelope's transform.

The echo times come from the analysis, and the line each echo lands on from
the k-space it reports, so the envelope follows the sequence's own ordering.


In [ ]:
k_adc, _, t_excitation, _, t_adc = seq.calculate_kspacePP()
echo_time = t_adc.reshape(MATRIX, MATRIX).mean(axis=1) - t_excitation[0]
line = np.round(k_adc[1].reshape(MATRIX, MATRIX)[:, 0] * FOV).astype(int)
order = np.argsort(line)

T2_STARS = (20e-3, 40e-3, 60e-3, 100e-3)


def point_spread(t2_star):
    """The phase-encode point-spread function the decay produces."""
    envelope = np.exp(-echo_time[order] / t2_star)
    spread = np.abs(np.fft.fftshift(np.fft.fft(np.fft.ifftshift(envelope))))
    return spread / spread.max()


def width(spread):
    """Full width at half maximum, in pixels, by linear interpolation."""
    above = np.flatnonzero(spread >= 0.5)
    first, last = above[0], above[-1]
    left = np.interp(0.5, spread[first - 1 : first + 1], [first - 1, first])
    right = np.interp(0.5, spread[last : last + 2][::-1], [last + 1, last])
    return right - left


widths = {t2_star: width(point_spread(t2_star)) for t2_star in T2_STARS}

print(f"\n{'T2*':>8}  {'decay over the train':>22}  {'point spread':>14}")
for t2_star in T2_STARS:
    print(
        f"{1e3 * t2_star:5.0f} ms  "
        f"{np.exp(-echo_time[order][-1] / t2_star):22.3f}  "
        f"{widths[t2_star]:11.2f} px"
    )

figure, (envelope_axis, spread_axis) = plt.subplots(1, 2, figsize=(PAGE_WIDTH, 3.2))
pixels = np.arange(MATRIX) - MATRIX // 2
for t2_star in T2_STARS:
    label = f"{1e3 * t2_star:.0f} ms"
    envelope_axis.plot(
        line[order], np.exp(-echo_time[order] / t2_star), lw=1.2, label=label
    )
    spread_axis.plot(pixels, point_spread(t2_star), lw=1.2, label=label)
envelope_axis.set_xlabel("phase-encode line")
envelope_axis.set_ylabel("signal")
spread_axis.set_xlim(-8, 8)
spread_axis.set_xlabel("pixels along the phase-encode direction")
spread_axis.set_ylabel("point spread, normalised")
envelope_axis.legend(
    frameon=False,
    title=r"$T_2^*$",
    loc="upper left",
    bbox_to_anchor=(0.0, 1.34),
    ncols=4,
    fontsize=9,
    title_fontsize=9,
)
figure.tight_layout(rect=(0, 0, 1, 0.86))

The envelope is not centred on the middle of k-space: the train runs from one
edge to the other, so the decay is monotonic across the lines rather than
symmetric about the line the echo is on. What that produces is a point-spread
function that is both wider than one pixel and asymmetric, which is the
blurring along the phase-encode direction an echo planar image carries. At
the shortest $T_2^*$ here the signal at the last line is a tenth of
the first and the point spread is half again as wide as a pixel; at the
longest it is within a tenth of a pixel of the unblurred width.

The remedies are the ones the previous page measured — a shorter echo
spacing, or fewer lines per shot — together with acquiring fewer lines
outright, by partial Fourier or by parallel imaging.


## Gradient delay and the odd echoes

A delay between the gradient waveform and the acquisition displaces every
sample along the readout direction by the distance k-space travels in that
delay. The readouts alternate in polarity, so the displacement alternates in
sign: the odd and the even lines of the matrix are shifted in opposite
directions.

:meth:`~pypulseqpp.Sequence.calculate_kspacePP` takes the delay as a
parameter, so the displacement is measured from the trajectory the analysis
reports rather than computed beside it.


In [ ]:
DELAYS = np.array([0.0, 1e-6, 2e-6, 4e-6, 8e-6])

displacement = []
for delay in DELAYS:
    delayed = seq.calculate_kspacePP(trajectory_delay=delay)[0]
    kx = delayed[0].reshape(MATRIX, MATRIX)
    # The echo of each line, where the ideal trajectory crosses zero.
    centre = kx[:, MATRIX // 2] * FOV
    displacement.append(
        {
            "delay": delay,
            "samples": float(np.abs(centre[::2].mean() - centre[1::2].mean()) / 2),
        }
    )

print(f"\n{'delay':>10}  {'odd-even displacement':>24}")
for row in displacement:
    print(f"{1e6 * row['delay']:7.1f} us  {row['samples']:20.3f} samples")

figure, axis = plt.subplots(figsize=(PAGE_WIDTH * 0.58, 3.0))
axis.plot(
    1e6 * DELAYS,
    [row["samples"] for row in displacement],
    "o-",
    lw=1.2,
    ms=5,
    label="displacement of a line from the ideal",
)
axis.set_xlabel("gradient delay (us)")
axis.set_ylabel("samples")
axis.legend(frameon=False, loc="upper left", bbox_to_anchor=(0.0, 1.26), fontsize=9)
figure.tight_layout(rect=(0, 0, 1, 0.88))

The displacement is the delay divided by the dwell time, and it is the same
for every line, so what distinguishes the odd lines from the even ones is its
sign. A quantity that alternates from one line to the next along the
phase-encode direction is, after the transform, an image displaced by half
the field of view, which is the ghost an uncorrected echo planar acquisition
shows. Measuring the delay and correcting for it belongs to the
reconstruction, which is Pulserver's part of the stack rather than this
package's.
